# Configuración del Entorno e Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import warnings
import time
import json
import os

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report, 
                             roc_auc_score, roc_curve, average_precision_score, 
                             precision_recall_curve, f1_score, recall_score, 
                             accuracy_score)

from mealpy import FloatVar, GWO, MFO
from scipy.stats import wilcoxon, shapiro, t as t_dist

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", context="paper")

# ============================================================
# REPRODUCIBILIDAD GLOBAL
# ============================================================
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

# Directorio de salida para figuras y resultados
OUTPUT_DIR = "resultados_gwo_mfo"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"XGBoost version: {xgb.__version__}")
print(f"Directorio de salida: {OUTPUT_DIR}")

# Carga y Preprocesamiento de Datos

In [ ]:
df = pd.read_csv('../../../Dataset/default of credit card clients.csv', sep=',', skiprows=1)

if 'default payment next month' in df.columns:
    df.rename(columns={'default payment next month': 'target'}, inplace=True)

if 'ID' in df.columns:
    df.drop('ID', axis=1, inplace=True)

print(f"Dimensiones del dataset: {df.shape}")
print(f"Distribución de clases:\n{df['target'].value_counts()}")
print(f"Prevalencia clase positiva: {df['target'].mean():.4f}")

# División Train/Test y Cálculo de Scale Pos Weight

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=GLOBAL_SEED, stratify=y
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
base_scale_pos_weight = neg / pos

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Clase 0 (train): {neg} | Clase 1 (train): {pos}")
print(f"Scale Pos Weight: {base_scale_pos_weight:.4f}")

# Función de Evaluación Completa (con TP/FP/TN/FN)

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name="Modelo"):
    """Evalúa un modelo y retorna todas las métricas + matriz de confusión."""
    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"--- {model_name} ---")
    print(f"PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc_auc:.4f} | "
          f"F1: {f1:.4f} | Recall: {rec:.4f} | Acc: {acc:.4f}")
    print(f"TP={tp} | FP={fp} | TN={tn} | FN={fn}")
    
    return {
        "pr_auc": pr_auc, "roc_auc": roc_auc, "f1": f1,
        "recall": rec, "accuracy": acc,
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)
    }

# Entrenamiento del Modelo Base XGBoost

In [ ]:
t0 = time.time()
model_base = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=base_scale_pos_weight,
    tree_method='hist', device='cuda', eval_metric='aucpr',
    random_state=GLOBAL_SEED, verbosity=0
)
model_base.fit(X_train, y_train, verbose=False)
t_base = time.time() - t0

y_pred_base = model_base.predict(X_test)
y_prob_base = model_base.predict_proba(X_test)[:, 1]
metrics_base = evaluate_model(y_test, y_pred_base, y_prob_base, "MODELO BASE")
print(f"⏱️ Tiempo entrenamiento base: {t_base:.1f}s")

# Visualización del Modelo Base

In [ ]:
def plot_model_evaluation(y_true, y_pred, y_prob, title_prefix="Modelo", 
                          save_path=None):
    """Genera matriz de confusión, ROC y PR. Opcionalmente guarda."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Cumple (0)', 'Incumple (1)'],
                yticklabels=['Cumple (0)', 'Incumple (1)'])
    axes[0].set_title(f'{title_prefix} - Matriz de Confusión')
    axes[0].set_ylabel('Valor Real')
    axes[0].set_xlabel('Predicción')

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)
    axes[1].plot(fpr, tpr, color='darkorange', lw=2, 
                 label=f'ROC (AUC = {roc_auc:.4f})')
    axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[1].set_xlabel('Tasa de Falsos Positivos')
    axes[1].set_ylabel('Tasa de Verdaderos Positivos')
    axes[1].set_title(f'{title_prefix} - Curva ROC')
    axes[1].legend(loc="lower right")

    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    axes[2].plot(recall, precision, color='green', lw=2, 
                 label=f'PR (AP = {pr_auc:.4f})')
    axes[2].set_xlabel('Recall (Sensibilidad)')
    axes[2].set_ylabel('Precision')
    axes[2].set_title(f'{title_prefix} - Curva Precision-Recall')
    axes[2].legend(loc="lower left")
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"💾 Figura guardada: {save_path}")
    plt.show()

plot_model_evaluation(y_test, y_pred_base, y_prob_base, "Base XGBoost",
                      save_path=f"{OUTPUT_DIR}/eval_BASE.png")

# Función Objetivo y Límites para GWO

In [ ]:
LB = [100, 3, 0.01, 0.6, 0.6, 2.0]
UB = [600, 10, 0.15, 1.0, 1.0, 5.0]

MIN_RECALL_THRESHOLD = 0.55
PENALTY_LAMBDA = 0.8  # ← λ = 0.8 (NO 10)

def extract_params(solution):
    """Extrae hiperparámetros con redondeo correcto (P-1)."""
    params = {
        "n_estimators": int(round(solution[0])),
        "max_depth": int(round(solution[1])),
        "learning_rate": float(solution[2]),
        "subsample": float(solution[3]),
        "colsample_bytree": float(solution[4]),
        "scale_pos_weight": float(solution[5])
    }
    # P-7: Verificación de rango
    for i, (key, lb, ub) in enumerate(zip(
        ["n_estimators","max_depth","learning_rate",
         "subsample","colsample_bytree","scale_pos_weight"], LB, UB)):
        val = params[key]
        if not (lb <= val <= ub):
            print(f"⚠️ {key}={val} fuera de [{lb},{ub}], clampeando.")
            params[key] = max(lb, min(ub, val))
    return params

def make_objective(seed_val):
    """Crea función objetivo con semilla específica (P-8: fuera del loop)."""
    def obj(solution):
        params = extract_params(solution)
        params.update({
            "tree_method": 'hist', "device": 'cuda', "eval_metric": 'aucpr',
            "early_stopping_rounds": 30, "random_state": seed_val, "verbosity": 0
        })
        
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed_val)
        scores = []
        
        for train_idx, val_idx in skf.split(X_train, y_train):
            X_tr = X_train.iloc[train_idx]
            X_val = X_train.iloc[val_idx]
            y_tr = y_train.iloc[train_idx]
            y_val = y_train.iloc[val_idx]
            
            m = xgb.XGBClassifier(**params)
            m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            
            y_p = m.predict(X_val)
            y_p_prob = m.predict_proba(X_val)[:, 1]
            
            pr_auc = average_precision_score(y_val, y_p_prob)
            f1 = f1_score(y_val, y_p)
            rec = recall_score(y_val, y_p)
            
            combined = np.sqrt(pr_auc * f1)  # Media geométrica
            penalty = max(0.0, (MIN_RECALL_THRESHOLD - rec) * PENALTY_LAMBDA)
            scores.append((1.0 - combined) + penalty)
            
        return np.mean(scores)
    return obj

# Función objetivo principal (semilla fija = GLOBAL_SEED)
objective_function_gwo = make_objective(GLOBAL_SEED)

problem_dict = {
    "obj_func": objective_function_gwo,
    "bounds": FloatVar(lb=LB, ub=UB, name="xgb_hyperparams"),
    "minmax": "min",
}

# Optimización GWO (20 Iteraciones)

In [ ]:
np.random.seed(GLOBAL_SEED)  # P-9: Semilla explícita para Mealpy
t0 = time.time()
gwo_20_model = GWO.OriginalGWO(epoch=20, pop_size=20)
gwo_20_best = gwo_20_model.solve(problem_dict)
t_gwo20 = time.time() - t0

print(f"✅ GWO-20 completado en {t_gwo20:.1f}s")
print(f"Mejor fitness: {gwo_20_best.target.fitness:.4f}")
print(f"OFEs reales: 20 × 20 × 3 = {20*20*3}")

best_params_20 = extract_params(gwo_20_best.solution)
print(f"Mejores hiperparámetros: {best_params_20}")

# Entrenamiento Final GWO (20 Iteraciones)

In [ ]:
model_gwo_20 = xgb.XGBClassifier(
    **best_params_20, tree_method='hist', device='cuda',
    eval_metric='aucpr', random_state=GLOBAL_SEED, verbosity=0
)
model_gwo_20.fit(X_train, y_train, verbose=False)

y_pred_gwo_20 = model_gwo_20.predict(X_test)
y_prob_gwo_20 = model_gwo_20.predict_proba(X_test)[:, 1]
metrics_gwo_20 = evaluate_model(y_test, y_pred_gwo_20, y_prob_gwo_20, "GWO (20)")

plot_model_evaluation(y_test, y_pred_gwo_20, y_prob_gwo_20, "GWO (20 Iter)",
                      save_path=f"{OUTPUT_DIR}/eval_GWO_20.png")

# Optimización GWO (50 Iteraciones)

In [ ]:
np.random.seed(GLOBAL_SEED)
t0 = time.time()
gwo_50_model = GWO.OriginalGWO(epoch=50, pop_size=20)
gwo_50_best = gwo_50_model.solve(problem_dict)
t_gwo50 = time.time() - t0

print(f"✅ GWO-50 completado en {t_gwo50:.1f}s")
print(f"Mejor fitness: {gwo_50_best.target.fitness:.4f}")
print(f"OFEs reales: 50 × 20 × 3 = {50*20*3}")

best_params_50 = extract_params(gwo_50_best.solution)
print(f"Mejores hiperparámetros: {best_params_50}")

# Entrenamiento Final GWO (50 Iteraciones)

In [ ]:
model_gwo_50 = xgb.XGBClassifier(
    **best_params_50, tree_method='hist', device='cuda',
    eval_metric='aucpr', random_state=GLOBAL_SEED, verbosity=0
)
model_gwo_50.fit(X_train, y_train, verbose=False)

y_pred_gwo_50 = model_gwo_50.predict(X_test)
y_prob_gwo_50 = model_gwo_50.predict_proba(X_test)[:, 1]
metrics_gwo_50 = evaluate_model(y_test, y_pred_gwo_50, y_prob_gwo_50, "GWO (50)")

plot_model_evaluation(y_test, y_pred_gwo_50, y_prob_gwo_50, "GWO (50 Iter)",
                      save_path=f"{OUTPUT_DIR}/eval_GWO_50.png")

# Optimización Híbrida GWO + MFO (20 + 20)

In [ ]:
def run_hybrid(gwo_epochs, mfo_epochs, pop_size, seed, label=""):
    """Ejecuta la estrategia híbrida GWO→MFO con todas las verificaciones."""
    np.random.seed(seed)
    
    # FASE 1: GWO Global
    print(f"🐺 FASE 1: GWO ({gwo_epochs} épocas) - {label}")
    t0 = time.time()
    gwo = GWO.OriginalGWO(epoch=gwo_epochs, pop_size=pop_size)
    gwo_best = gwo.solve(problem_dict)
    t_gwo = time.time() - t0
    print(f"   Fitness GWO: {gwo_best.target.fitness:.4f} ({t_gwo:.1f}s)")
    
    # FASE 2: MFO Local
    best_gwo = gwo_best.solution
    margin = 0.20
    LB_loc = [max(lb, b - (ub - lb) * margin) for lb, ub, b in zip(LB, UB, best_gwo)]
    UB_loc = [min(ub, b + (ub - lb) * margin) for lb, ub, b in zip(LB, UB, best_gwo)]
    
    # P-17: Verificar que LB < UB
    for i, (l, u) in enumerate(zip(LB_loc, UB_loc)):
        if l >= u:
            print(f"⚠️ LB_loc[{i}]={l:.4f} >= UB_loc[{i}]={u:.4f}, ajustando.")
            mid = (l + u) / 2
            LB_loc[i] = mid - 0.01
            UB_loc[i] = mid + 0.01
    
    print(f"🦋 FASE 2: MFO ({mfo_epochs} épocas) - {label}")
    print(f"   LB_local: {np.round(LB_loc, 4)}")
    print(f"   UB_local: {np.round(UB_loc, 4)}")
    
    prob_local = {
        "obj_func": make_objective(seed),
        "bounds": FloatVar(lb=LB_loc, ub=UB_loc, name="xgb_local"),
        "minmax": "min",
    }
    
    np.random.seed(seed + 1)  # Semilla diferente para MFO
    t0 = time.time()
    mfo = MFO.OriginalMFO(epoch=mfo_epochs, pop_size=pop_size)
    mfo_best = mfo.solve(prob_local)
    t_mfo = time.time() - t0
    print(f"   Fitness MFO: {mfo_best.target.fitness:.4f} ({t_mfo:.1f}s)")
    
    ofe_total = (gwo_epochs + mfo_epochs) * pop_size * 3
    print(f"   OFEs totales: ({gwo_epochs}+{mfo_epochs}) × {pop_size} × 3 = {ofe_total}")
    
    return extract_params(mfo_best.solution), t_gwo + t_mfo, ofe_total

# Híbrido 20+20
best_params_hybrid_20, t_hyb20, ofe_hyb20 = run_hybrid(
    gwo_epochs=20, mfo_epochs=20, pop_size=20, 
    seed=GLOBAL_SEED, label="Híbrido (20+20)"
)
print(f"Mejores hiperparámetros Híbrido (20+20): {best_params_hybrid_20}")

# Entrenamiento Final Híbrido (20+20)

In [ ]:
model_hybrid_20 = xgb.XGBClassifier(
    **best_params_hybrid_20, tree_method='hist', device='cuda',
    eval_metric='aucpr', random_state=GLOBAL_SEED, verbosity=0
)
model_hybrid_20.fit(X_train, y_train, verbose=False)

y_pred_hybrid_20 = model_hybrid_20.predict(X_test)
y_prob_hybrid_20 = model_hybrid_20.predict_proba(X_test)[:, 1]
metrics_hybrid_20 = evaluate_model(
    y_test, y_pred_hybrid_20, y_prob_hybrid_20, "Híbrido (20+20)"
)

plot_model_evaluation(y_test, y_pred_hybrid_20, y_prob_hybrid_20, 
                      "Híbrido GWO+MFO (20+20)",
                      save_path=f"{OUTPUT_DIR}/eval_HYB_20_20.png")

# Optimización Híbrida GWO + MFO (50 + 20)

In [ ]:
best_params_hybrid_50, t_hyb50, ofe_hyb50 = run_hybrid(
    gwo_epochs=50, mfo_epochs=20, pop_size=20,
    seed=GLOBAL_SEED, label="Híbrido (50+20)"
)
print(f"Mejores hiperparámetros Híbrido (50+20): {best_params_hybrid_50}")

# Entrenamiento Final Híbrido (50+20)

In [ ]:
model_hybrid_50 = xgb.XGBClassifier(
    **best_params_hybrid_50, tree_method='hist', device='cuda',
    eval_metric='aucpr', random_state=GLOBAL_SEED, verbosity=0
)
model_hybrid_50.fit(X_train, y_train, verbose=False)

y_pred_hybrid_50 = model_hybrid_50.predict(X_test)
y_prob_hybrid_50 = model_hybrid_50.predict_proba(X_test)[:, 1]
metrics_hybrid_50 = evaluate_model(
    y_test, y_pred_hybrid_50, y_prob_hybrid_50, "Híbrido (50+20)"
)

plot_model_evaluation(y_test, y_pred_hybrid_50, y_prob_hybrid_50,
                      "Híbrido GWO+MFO (50+20)",
                      save_path=f"{OUTPUT_DIR}/eval_HYB_50_20.png")

# Tabla Comparativa de Rendimiento

In [ ]:
all_metrics = {
    "XGBoost Base": metrics_base,
    "GWO (20 Iter)": metrics_gwo_20,
    "GWO (50 Iter)": metrics_gwo_50,
    "Híbrido (20+20)": metrics_hybrid_20,
    "Híbrido (50+20)": metrics_hybrid_50,
}

comparison_df = pd.DataFrame(all_metrics).T
comparison_df["Geo-Mean"] = np.sqrt(comparison_df["pr_auc"] * comparison_df["f1"])
comparison_df["Δ PR-AUC (%)"] = ((comparison_df["pr_auc"] / metrics_base["pr_auc"]) - 1) * 100

print("\n📊 TABLA COMPARATIVA")
print(comparison_df[["pr_auc","f1","recall","accuracy","Geo-Mean","Δ PR-AUC (%)"]].to_string())

# P-23: Exportar para el artículo
comparison_df.to_csv(f"{OUTPUT_DIR}/tabla_comparativa.csv")
print(f"\n💾 Tabla exportada: {OUTPUT_DIR}/tabla_comparativa.csv")

# Análisis de Robustez Estadística (15 ejecuciones)

In [ ]:
seeds = [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 
         2021, 2223, 2425, 2627, 2829, 3031]

results = {
    "base": [], "gwo_20": [], "gwo_50": [], 
    "hybrid_20": [], "hybrid_50": []
}

print(f"Iniciando análisis de robustez ({len(seeds)} ejecuciones)...")
t_robust_start = time.time()

for i, seed in enumerate(seeds):
    print(f"\n{'='*60}")
    print(f"Ejecución {i+1}/{len(seeds)} (Seed: {seed})")
    print(f"{'='*60}")
    
    # --- Modelo Base ---
    mb = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=base_scale_pos_weight,
        tree_method='hist', device='cuda', eval_metric='aucpr',
        random_state=seed, verbosity=0
    )
    mb.fit(X_train, y_train, verbose=False)
    results["base"].append(
        average_precision_score(y_test, mb.predict_proba(X_test)[:, 1])
    )
    
    # --- GWO 20 ---
    np.random.seed(seed)
    obj_seed = make_objective(seed)
    prob_20 = {"obj_func": obj_seed, 
               "bounds": FloatVar(lb=LB, ub=UB, name="x"), "minmax": "min"}
    g20 = GWO.OriginalGWO(epoch=20, pop_size=20)
    best_g20 = g20.solve(prob_20)
    bp_g20 = extract_params(best_g20.solution)
    bp_g20.update({"tree_method": 'hist', "device": 'cuda', 
                   "eval_metric": 'aucpr', "random_state": seed, "verbosity": 0})
    m_g20 = xgb.XGBClassifier(**bp_g20)
    m_g20.fit(X_train, y_train, verbose=False)  # P-10: verbose=False
    results["gwo_20"].append(
        average_precision_score(y_test, m_g20.predict_proba(X_test)[:, 1])
    )
    
    # --- GWO 50 ---
    np.random.seed(seed)
    g50 = GWO.OriginalGWO(epoch=50, pop_size=20)
    best_g50 = g50.solve(prob_20)
    bp_g50 = extract_params(best_g50.solution)
    bp_g50.update({"tree_method": 'hist', "device": 'cuda',
                   "eval_metric": 'aucpr', "random_state": seed, "verbosity": 0})
    m_g50 = xgb.XGBClassifier(**bp_g50)
    m_g50.fit(X_train, y_train, verbose=False)
    results["gwo_50"].append(
        average_precision_score(y_test, m_g50.predict_proba(X_test)[:, 1])
    )
    
    # --- Híbrido 20+20 ---
    best_sol_20 = best_g20.solution
    LB_l = [max(lb, b - (ub-lb)*0.20) for lb, ub, b in zip(LB, UB, best_sol_20)]
    UB_l = [min(ub, b + (ub-lb)*0.20) for lb, ub, b in zip(LB, UB, best_sol_20)]
    for j, (l, u) in enumerate(zip(LB_l, UB_l)):
        if l >= u:
            mid = (l + u) / 2; LB_l[j] = mid - 0.01; UB_l[j] = mid + 0.01
    prob_mfo_20 = {"obj_func": obj_seed,
                   "bounds": FloatVar(lb=LB_l, ub=UB_l, name="x"), "minmax": "min"}
    np.random.seed(seed + 1)
    mfo_20 = MFO.OriginalMFO(epoch=20, pop_size=20)
    best_m20 = mfo_20.solve(prob_mfo_20)
    bp_m20 = extract_params(best_m20.solution)
    bp_m20.update({"tree_method": 'hist', "device": 'cuda',
                   "eval_metric": 'aucpr', "random_state": seed, "verbosity": 0})
    m_m20 = xgb.XGBClassifier(**bp_m20)
    m_m20.fit(X_train, y_train, verbose=False)
    results["hybrid_20"].append(
        average_precision_score(y_test, m_m20.predict_proba(X_test)[:, 1])
    )
    
    # --- Híbrido 50+20 ---
    best_sol_50 = best_g50.solution
    LB_l50 = [max(lb, b - (ub-lb)*0.20) for lb, ub, b in zip(LB, UB, best_sol_50)]
    UB_l50 = [min(ub, b + (ub-lb)*0.20) for lb, ub, b in zip(LB, UB, best_sol_50)]
    for j, (l, u) in enumerate(zip(LB_l50, UB_l50)):
        if l >= u:
            mid = (l + u) / 2; LB_l50[j] = mid - 0.01; UB_l50[j] = mid + 0.01
    prob_mfo_50 = {"obj_func": obj_seed,
                   "bounds": FloatVar(lb=LB_l50, ub=UB_l50, name="x"), "minmax": "min"}
    np.random.seed(seed + 1)
    mfo_50 = MFO.OriginalMFO(epoch=20, pop_size=20)
    best_m50 = mfo_50.solve(prob_mfo_50)
    bp_m50 = extract_params(best_m50.solution)
    bp_m50.update({"tree_method": 'hist', "device": 'cuda',
                   "eval_metric": 'aucpr', "random_state": seed, "verbosity": 0})
    m_m50 = xgb.XGBClassifier(**bp_m50)
    m_m50.fit(X_train, y_train, verbose=False)
    results["hybrid_50"].append(
        average_precision_score(y_test, m_m50.predict_proba(X_test)[:, 1])
    )
    
    # P-11: Checkpoint cada 5 ejecuciones
    if (i + 1) % 5 == 0:
        with open(f"{OUTPUT_DIR}/robustness_checkpoint.json", "w") as f:
            json.dump({"seeds": seeds[:i+1], "results": results}, f, indent=2)
        print(f"💾 Checkpoint guardado (ejecución {i+1})")

t_robust_total = time.time() - t_robust_start
print(f"\n✅ Análisis de robustez completado en {t_robust_total/3600:.2f} horas")

# Guardado final
with open(f"{OUTPUT_DIR}/robustness_results.json", "w") as f:
    json.dump({"seeds": seeds, "results": results, 
               "total_time_hours": t_robust_total/3600}, f, indent=2)
print(f"💾 Resultados guardados: {OUTPUT_DIR}/robustness_results.json")

# Análisis Estadístico Completo (IC 95% + Wilcoxon + Holm-Bonferroni)

In [ ]:
# ============================================================
# INTERVALOS DE CONFIANZA 95%
# ============================================================
def ci_95(data):
    n = len(data)
    m = np.mean(data)
    se = np.std(data, ddof=1) / np.sqrt(n)
    t_crit = t_dist.ppf(0.975, df=n-1)
    return m, m - t_crit * se, m + t_crit * se

print("\n📊 INTERVALOS DE CONFIANZA (95%)")
print(f"{'Modelo':25s} {'Media':>8s} {'IC_inf':>8s} {'IC_sup':>8s} {'Var':>12s}")
print("-" * 65)
for name, key in [("XGBoost Base", "base"), ("GWO (20)", "gwo_20"),
                   ("GWO (50)", "gwo_50"), ("Híbrido (20+20)", "hybrid_20"),
                   ("Híbrido (50+20)", "hybrid_50")]:
    data = results[key]
    m, lo, hi = ci_95(data)
    v = np.var(data, ddof=1)
    print(f"{name:25s} {m:8.4f} {lo:8.4f} {hi:8.4f} {v:12.2e}")

# ============================================================
# SHAPIRO-WILK (justificación de test no paramétrico)
# ============================================================
print("\n📊 SHAPIRO-WILK (normalidad)")
for name, key in [("Base", "base"), ("GWO-20", "gwo_20"), 
                   ("GWO-50", "gwo_50"), ("Híb-20+20", "hybrid_20"),
                   ("Híb-50+20", "hybrid_50")]:
    stat_sw, p_sw = shapiro(results[key])
    normal = "Normal" if p_sw > 0.05 else "No normal"
    print(f"  {name:15s}: W={stat_sw:.4f}, p={p_sw:.4f} → {normal}")

# ============================================================
# WILCOXON + HOLM-BONFERRONI + EFFECT SIZE
# ============================================================
comparisons = [
    ("Híb(20+20) vs Base",       "hybrid_20", "base"),
    ("Híb(50+20) vs Base",       "hybrid_50", "base"),
    ("Híb(20+20) vs GWO(20)",    "hybrid_20", "gwo_20"),
    ("Híb(50+20) vs GWO(50)",    "hybrid_50", "gwo_50"),
    ("Híb(50+20) vs Híb(20+20)", "hybrid_50", "hybrid_20"),
]

n_comp = len(comparisons)
n = len(seeds)
max_W = n * (n + 1) / 2  # = 120

wilcoxon_results = []
for name, key_a, key_b in comparisons:
    W, p = wilcoxon(results[key_a], results[key_b], alternative='greater')
    r_rb = 1 - (2 * W) / max_W
    wilcoxon_results.append((name, W, p, r_rb))

# Holm-Bonferroni: ordenar por p-value
wilcoxon_results.sort(key=lambda x: x[2])

print(f"\n{'='*90}")
print("WILCOXON + HOLM-BONFERRONI + EFFECT SIZE")
print(f"{'='*90}")
print(f"{'Comparación':35s} {'W':>5s} {'p':>10s} {'p_Holm':>10s} {'r_rb':>7s} {'Sig?':>5s}")
print("-" * 90)

for rank, (name, W, p, r_rb) in enumerate(wilcoxon_results):
    p_holm = min(p * (n_comp - rank), 1.0)
    sig = "✅" if p_holm < 0.05 else "❌"
    print(f"{name:35s} {W:5.0f} {p:10.6f} {p_holm:10.6f} {r_rb:7.3f} {sig:>5s}")

# Potencia estadística con n=15
print(f"\n📊 Potencia estadística: n={n}, p_min posible = {1/2**n:.6f}")
print(f"   Con α=0.05 y n=15, se necesita W≥115 para significancia unilateral.")

# Boxplots PR-AUC

In [ ]:
plt.figure(figsize=(14, 6))

df_robust = pd.DataFrame({
    "XGBoost Base": results["base"],
    "GWO (20 Iter)": results["gwo_20"],
    "GWO (50 Iter)": results["gwo_50"],
    "Híbrido (20+20)": results["hybrid_20"],
    "Híbrido (50+20)": results["hybrid_50"]
}).melt(var_name="Modelo", value_name="PR-AUC")

ax = sns.boxplot(data=df_robust, x="Modelo", y="PR-AUC", palette="Set2",
                 linewidth=1.5, fliersize=5)
sns.stripplot(data=df_robust, x="Modelo", y="PR-AUC", color="black",
              size=4, alpha=0.6, jitter=True)

models_list = df_robust["Modelo"].unique()
means_list = [df_robust[df_robust["Modelo"] == m]["PR-AUC"].mean() for m in models_list]
vars_list = [df_robust[df_robust["Modelo"] == m]["PR-AUC"].var() for m in models_list]

for i, (model, mean) in enumerate(zip(models_list, means_list)):
    ax.plot(i, mean, marker='D', color='red', markersize=8,
            label='Media' if i == 0 else "")

plt.title("Análisis de Robustez: Distribución de PR-AUC (15 Ejecuciones)",
          fontsize=14, fontweight='bold')
plt.ylabel("PR-AUC", fontsize=12)
plt.xlabel("Configuración", fontsize=12)
plt.ylim(min(df_robust["PR-AUC"]) - 0.01, max(df_robust["PR-AUC"]) + 0.01)
plt.legend(title="Estadístico", loc="lower right")
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, (model, var, mean_val) in enumerate(zip(models_list, vars_list, means_list)):
    var_text = f'Var: {var:.2e}' if var < 0.00001 else f'Var: {var:.6f}'
    ax.annotate(var_text, xy=(i, mean_val), 
                xytext=(i, max(df_robust["PR-AUC"]) + 0.003),
                ha='center', fontsize=9, color='darkred', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Boxplot_PR-AUC_15_Ejecuciones.png", 
            dpi=300, bbox_inches='tight')
print(f"💾 Boxplot guardado: {OUTPUT_DIR}/Boxplot_PR-AUC_15_Ejecuciones.png")
plt.show()

# Resumen de Tiempos y OFEs

In [ ]:
print("\n📊 RESUMEN DE TIEMPOS Y OFEs")
print(f"{'Método':25s} {'Tiempo (s)':>12s} {'OFEs':>8s}")
print("-" * 50)
print(f"{'GWO (20 iter)':25s} {t_gwo20:12.1f} {20*20*3:8d}")
print(f"{'GWO (50 iter)':25s} {t_gwo50:12.1f} {50*20*3:8d}")
print(f"{'Híbrido (20+20)':25s} {t_hyb20:12.1f} {ofe_hyb20:8d}")
print(f"{'Híbrido (50+20)':25s} {t_hyb50:12.1f} {ofe_hyb50:8d}")
print(f"{'Robustez (total)':25s} {t_robust_total:12.1f} {'N/A':>8s}")